In [2]:
import os
if os.getcwd().endswith('notebooks'):
    print('here')
    os.chdir(r'..')
    os.chdir(r'..')
    os.chdir(r'chess_engine')
import sys
sys.path.append('./')
# os.chdir(r'..\..\..\chess_engine')
print(os.getcwd())
from chess_engine.src.model.classes.autoencoder.AE_DataLoader import get_dataloader, FlattenTransform
from chess_engine.src.model.config.config import data_settings
import torch
import torch.nn as nn
import torch.nn.functional as F
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import sample_bitboard_dict
from chess_engine.src.model.classes.autoencoder.feature_extractor import sample_metada

C:\Users\ethan\git\Full_Chess_App\chess_engine


ModuleNotFoundError: No module named 'chess_engine.src.model.classes.autoencoder.AE_Dataloader'

In [16]:
len(sample_metada) + len(sample_bitboard_dict)*8*8

836

In [2]:
transform = FlattenTransform()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def get_dataloaders(transform):
    train_loader = get_dataloader(data_settings.TrainingDirectory,
                              batch_size=64,
                              shuffle=True,
                              num_workers=0,
                              transform=transform)
    test_loader = get_dataloader(data_settings.TestingDirectory,
                                  batch_size=64,
                                  shuffle=True,
                                  num_workers=0,
                                  transform=transform)
    valid_loader = get_dataloader(data_settings.ValidationDirectory,
                                  batch_size=64,
                                  shuffle=True,
                                  num_workers=0,
                                  transform=transform)
    return train_loader, test_loader, valid_loader

In [ ]:
train_loader, test_loader, valid_loader = get_dataloaders(transform=None)

class MultiInputAutoencoder(nn.Module):
    def __init__(self, 
                 bitboard_in_channels=13, 
                 metadata_in_features=4, 
                 latent_dim=128):
        """
        :param bitboard_in_channels: Number of channels for bitboard input, e.g. 13
        :param metadata_in_features: Metadata dimension, e.g. 4
        :param latent_dim: Dimension of the shared latent vector
        """
        super(MultiInputAutoencoder, self).__init__()

        ## ----------------------
        ## Encoder for bitboards
        ## ----------------------
        self.bb_encoder = nn.Sequential(
            nn.Conv2d(bitboard_in_channels, 32, kernel_size=3, stride=2, padding=1),  # [B, 32, 4, 4]
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),                   # [B, 64, 2, 2]
            nn.ReLU(inplace=True),
            nn.Flatten(),  # [B, 64 * 2 * 2] = [B, 256]
            nn.Linear(64*2*2, 64),
            nn.ReLU(inplace=True),
        )

        ## ----------------------
        ## Encoder for metadata
        ## ----------------------
        self.meta_encoder = nn.Sequential(
            nn.Linear(metadata_in_features, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 32),
            nn.ReLU(inplace=True),
        )

        ## ----------------------
        ## Merge and produce latent
        ## ----------------------
        # The combined dimension from both encoders: 64 (bitboard) + 32 (metadata) = 96
        self.latent_fc = nn.Sequential(
            nn.Linear(96, latent_dim),
            nn.ReLU(inplace=True),
        )

        ## ----------------------
        ## Decoder for bitboards
        ## ----------------------
        # We'll decode from latent_dim back to the shape (13, 8, 8).
        # A common approach is to map latent_dim -> some flattened shape -> deconv
        # or just use MLP to get back to 13*8*8, then reshape.
        self.bb_decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 64*2*2),
            nn.ReLU(inplace=True)
        )
        self.bb_decoder_deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # [B,32,4,4]
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, bitboard_in_channels, kernel_size=3, stride=2, padding=1, 
                               output_padding=1),   # [B, 13, 8, 8]
            # If you expect the final data to be in [-1,1] or [0,1], you might use nn.Sigmoid() or nn.Tanh()
        )

        ## ----------------------
        ## Decoder for metadata
        ## ----------------------
        self.meta_decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, metadata_in_features)  # -> 4
            # Possibly no activation or ReLU/Tanh, depends on your metadata scale
        )

    def encode(self, bitboards, metadata):
        """
        bitboards shape: [B, 13, 8, 8]
        metadata shape:   [B, 4]
        """
        bb_code = self.bb_encoder(bitboards)       # => [B, 64]
        meta_code = self.meta_encoder(metadata)    # => [B, 32]
        combined = torch.cat([bb_code, meta_code], dim=1)  # => [B, 96]
        latent = self.latent_fc(combined)                 # => [B, latent_dim]
        return latent

    def decode(self, latent):
        """
        Splits into two decoders: one for bitboards, one for metadata
        """
        # Bitboard decode
        x_bb = self.bb_decoder_fc(latent)         # => [B, 64*2*2]
        x_bb = x_bb.view(-1, 64, 2, 2)            # reshape for deconv
        recon_bitboards = self.bb_decoder_deconv(x_bb)  # => [B, 13, 8, 8]

        # Metadata decode
        recon_metadata = self.meta_decoder(latent)  # => [B, 4]
        return recon_bitboards, recon_metadata

    def forward(self, bitboards, metadata):
        latent = self.encode(bitboards, metadata)
        return self.decode(latent)

model = MultiInputAutoencoder()

# Define your criterion (e.g., MSE) and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs=16
# Example training step
for epoch in range(num_epochs):
    for bitboards_batch, metadata_batc,_ in train_loader:
        optimizer.zero_grad()
        recon_bb, recon_meta = model(bitboards_batch, metadata_batch)
        
        # The bitboard reconstruction loss
        loss_bb = criterion(recon_bb, bitboards_batch)
        # The metadata reconstruction loss
        loss_meta = criterion(recon_meta, metadata_batch)
        
        # Total loss (you can weight them differently if desired)
        loss = loss_bb + loss_meta
        
        loss.backward()
        optimizer.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


In [4]:
class SingleInputAutoencoder(nn.Module):
    def __init__(self, input_dim=836, latent_dim=128):
        """
        :param input_dim: Size of flattened input features, e.g., 836
        :param latent_dim: Dimension of the latent space
        """
        super(SingleInputAutoencoder, self).__init__()
        self.latent_dim = latent_dim
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, latent_dim),
            # Optionally add an activation or not, depending on how you want your latent space
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, input_dim),
            # For final activation, if your data is normalized [0,1] you might do Sigmoid here
            # nn.Sigmoid()
        )

    def encode(self, x):
       return self.encoder(x)
    def decode(self, z):
       return self.decoder(z)
        
    def forward(self, x):
        """
        x shape: [B, 836]
        """
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

# autoenc = SingleInputAutoencoder()


# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# num_epochs = 1
# for epoch in range(num_epochs):
#     for x_batch,_ in train_loader:
#         # x_batch is shape [batch_size, 836]
#         # print(x_batch.shape)
#         optimizer.zero_grad()
#         x_recon = model(x_batch)
#         loss = criterion(x_recon, x_batch)
#         loss.backward()
#         optimizer.step()
#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


In [5]:
class FrozenEncoderWithNewLayers(nn.Module):

    """

    Uses a pretrained autoencoder's encoder as a frozen feature extractor,

    then adds new (trainable) layers on top for a new task (like in Deep Chess).

    """

    def __init__(self, pretrained_autoencoder, hidden_dim=64, output_dim=1):

        super().__init__()
        self.frozen_encoder = pretrained_autoencoder.encoder

        for param in self.frozen_encoder.parameters():
            param.requires_grad = False


        self.latent_dim = pretrained_autoencoder.latent_dim
        self.extra_layers = nn.Sequential(

            nn.Linear(self.latent_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)

        )



    def forward(self, x):
        with torch.no_grad():
            z = self.frozen_encoder(x)  # shape [B, latent_dim]
        out = self.extra_layers(z)     # shape [B, output_dim]
        return out

In [6]:


def train_autoencoder(model, train_loader, val_loader, num_epochs=5, lr=1e-3, device='cpu'):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for features_batch,_ in train_loader:
            features_batch = features_batch.to(device)
            optimizer.zero_grad()
            recon = model(features_batch)
            loss = criterion(recon, features_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * features_batch.size(0)
    
        train_loss = running_loss / len(train_loader.dataset)
        model.eval()
        val_loss = 0.0
    
        with torch.no_grad():
    
            for features_batch,_ in val_loader:
                features_batch = features_batch.to(device)
                recon = model(features_batch)
                loss = criterion(recon, features_batch)
                val_loss += loss.item() * features_batch.size(0)


                

    
        val_loss /= len(val_loader.dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}] - "
              f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [7]:

def train_new_head(model, train_loader, val_loader, num_epochs=5, lr=1e-3, device='cpu'):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for features_batch, labels_batch in train_loader:
            features_batch = features_batch.to(device)
            labels_batch = labels_batch.to(device)  # shape [B, 1] for this example
            optimizer.zero_grad()
            outputs = model(features_batch)         # shape [B, 1]
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * features_batch.size(0)
            
        train_loss = running_loss / len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
    
            for features_batch, labels_batch in val_loader:
                features_batch = features_batch.to(device)
                labels_batch = labels_batch.to(device)
                outputs = model(features_batch)
                loss = criterion(outputs, labels_batch)
                val_loss += loss.item() * features_batch.size(0)
                
                
        val_loss /= len(val_loader.dataset)
        print(f"NewHead Epoch [{epoch+1}/{num_epochs}] - "
              f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [10]:
def main():
    
    train_loader_ae, test_loader_ae, val_loader_ae = get_dataloaders(transform)



    autoenc = SingleInputAutoencoder(input_dim=836, latent_dim=128).to(device)
    print("Training Autoencoder...")
    train_autoencoder(autoenc, train_loader_ae, val_loader_ae, num_epochs=16, lr=1e-3, device=device)
    frozen_model = FrozenEncoderWithNewLayers(pretrained_autoencoder=autoenc,num_epochs=16,hidden_dim=64,output_dim=3).to(device)


    train_loader_ae, test_loader_ae, val_loader_ae = get_dataloaders(transform)
    
    print("\nTraining New Head on top of Frozen Encoder...")
    train_new_head(frozen_model, train_loader_ae, val_loader_ae, num_epochs=1, lr=1e-3, device=device)
    print("Done.")

In [11]:
main()

Training Autoencoder...
Epoch [1/16] - Train Loss: 0.0161, Val Loss: 0.0122
Epoch [2/16] - Train Loss: 0.0102, Val Loss: 0.0089
Epoch [3/16] - Train Loss: 0.0080, Val Loss: 0.0075
Epoch [4/16] - Train Loss: 0.0068, Val Loss: 0.0066
Epoch [5/16] - Train Loss: 0.0061, Val Loss: 0.0060
Epoch [6/16] - Train Loss: 0.0055, Val Loss: 0.0056
Epoch [7/16] - Train Loss: 0.0051, Val Loss: 0.0052
Epoch [8/16] - Train Loss: 0.0048, Val Loss: 0.0048
Epoch [9/16] - Train Loss: 0.0045, Val Loss: 0.0046
Epoch [10/16] - Train Loss: 0.0043, Val Loss: 0.0045
Epoch [11/16] - Train Loss: 0.0041, Val Loss: 0.0042
Epoch [12/16] - Train Loss: 0.0039, Val Loss: 0.0040
Epoch [13/16] - Train Loss: 0.0038, Val Loss: 0.0039
Epoch [14/16] - Train Loss: 0.0036, Val Loss: 0.0039
Epoch [15/16] - Train Loss: 0.0035, Val Loss: 0.0037
Epoch [16/16] - Train Loss: 0.0034, Val Loss: 0.0036


TypeError: FrozenEncoderWithNewLayers.__init__() got an unexpected keyword argument 'num_epochs'